# Ahmedabad House Rent Analysis and Prediction

## Project goal

In this project we will use the **Ahmedabad rent dataset** to:

1. Understand the dataset.
2. Clean the data.
3. Perform Exploratory Data Analysis (EDA).
4. Build a **Linear Regression** model.
5. Check the model using **MAE, RMSE and R²**.
6. Use the model to estimate rent for a new property.

### Important
This notebook is deliberately kept at the same level as the topics in the provided class notebooks.

We use only:
- Pandas
- NumPy
- Matplotlib
- Seaborn
- `LabelEncoder`
- `train_test_split`
- `LinearRegression`
- MAE, RMSE and R²
- Basic Streamlit concepts in the separate `app.py`

We do **not** use Random Forest, Gradient Boosting, pickle, joblib, one-hot encoding, pipelines, or other techniques that were not part of the provided lessons.

## Cell 1 — Import the libraries

We first import the libraries that we learned to use.

- **Pandas** → working with tables/data.
- **NumPy** → numerical calculations.
- **Matplotlib** → plotting graphs.
- **Seaborn** → easier and nicer statistical graphs.
- **warnings** → hides warning messages so the notebook is easier to read.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Hide warning messages
warnings.filterwarnings('ignore')

## Cell 2 — Read the dataset

`pd.read_csv()` reads the CSV file and stores it inside a DataFrame.

The DataFrame name is `rent`.

In [ ]:
rent = pd.read_csv('Ahmedabad_rent.csv')

# Display the first 5 rows
rent.head()

## Cell 3 — Look at the last rows

`tail()` displays the last 5 rows.

This helps us see whether the dataset looks consistent from beginning to end.

In [ ]:
rent.tail()

## Cell 4 — Understand the dataset

`info()` tells us:
- number of rows,
- column names,
- data types,
- and non-null values.

This is one of the first checks we should do with any dataset.

In [ ]:
rent.info()

## Cell 5 — Basic statistics

`describe()` gives statistics for numerical columns such as:
- count,
- mean,
- standard deviation,
- minimum,
- maximum.

At this stage the `price` column is still text because the dataset contains commas such as `20,000`.

In [ ]:
rent.describe()

## Cell 6 — Check missing values

`isnull().sum()` counts how many missing values are present in each column.

This is important before building a model.

In [ ]:
rent.isnull().sum()

## Cell 7 — Check data types

`dtypes` tells us the data type of every column.

In this dataset, `price` is stored as `object` because some values contain commas.

In [ ]:
rent.dtypes

## Cell 8 — Check duplicates

The dataset contains repeated rows. We use `drop_duplicates()` to keep one copy of an identical listing.

This is a simple data-cleaning step so the same exact row is not counted many times.

In [ ]:
print('Rows before removing duplicates:', len(rent))

rent = rent.drop_duplicates()

print('Rows after removing duplicates:', len(rent))

## Cell 9 — Clean the price column

The `price` column has values like:

- `20,000`
- `7,350`
- `1.1`

The commas make the column text, so first we remove commas and convert the column to `float`.

In this dataset, the small decimal values are lakh-format values. For example:

`1.1` means `₹1.1 lakh`, so we convert it to `110000`.

`np.where()` follows the type of conditional operation used in the lessons:

- if price is less than 100 → multiply by 100000
- otherwise keep the original value.

In [ ]:
# Remove commas from the price values
rent['price'] = rent['price'].str.replace(',', '')

# Convert price from text to number
rent['price'] = rent['price'].astype(float)

# Convert small lakh-format values to rupees
rent['price'] = np.where(
    rent['price'] < 100,
    rent['price'] * 100000,
    rent['price']
)

rent[['price']].head()

## Cell 10 — Clean the bathroom column

The `bathroom` column is not perfectly clean.

Normal values look like:
- `1 bathrooms`
- `2 bathrooms`
- `3 bathrooms`

But some rows contain values such as `North facing`, which is not a bathroom value.

We create a small function to:
- keep the number when the value contains `bathroom`,
- otherwise return a missing value.

Then we replace missing bathroom values with the median.

This uses the same simple function idea used in the class notebooks.

In [ ]:
# Function to convert bathroom text into a number
def get_bathroom(x):
    # If the value is already missing
    if pd.isnull(x):
        return np.nan

    # Keep the number if the text contains 'bathroom'
    if 'bathroom' in str(x):
        return float(str(x).split()[0])

    # Invalid values become missing
    return np.nan

# Apply the function to the bathroom column
rent['bathroom'] = rent['bathroom'].apply(get_bathroom)

# Replace missing bathroom values with the median
rent['bathroom'] = rent['bathroom'].fillna(rent['bathroom'].median())

rent[['bathroom']].head()

## Cell 11 — Check the cleaned data

Now the important columns are numeric where necessary.

We check:
- first few rows,
- data types,
- remaining missing values.

In [ ]:
rent.head()

print('\nData types:')
print(rent.dtypes)

print('\nMissing values:')
print(rent.isnull().sum())

## Cell 12 — Price distribution

A histogram helps us understand how rent values are distributed.

The graph can show whether most houses have lower rents and a smaller number have very high rents.

In [ ]:
plt.figure(figsize=(12,5))

sns.histplot(rent['price'], bins=20, kde=True, color='blue')

plt.title('Monthly Rent Distribution')
plt.xlabel('Monthly Rent')
plt.ylabel('Frequency')

plt.show()

## Cell 13 — Area distribution

This graph shows the distribution of house area in square feet.

In [ ]:
plt.figure(figsize=(12,5))

sns.histplot(rent['area'], bins=20, kde=True, color='green')

plt.title('Area Distribution')
plt.xlabel('Area (sq.ft)')
plt.ylabel('Frequency')

plt.show()

## Cell 14 — Number of bedrooms

`countplot()` shows how many listings belong to each bedroom category.

In [ ]:
plt.figure(figsize=(12,5))

sns.countplot(x='bedroom', data=rent, color='blue')

plt.title('Number of Bedrooms Distribution')
plt.xlabel('Bedrooms')
plt.ylabel('Number of Listings')

plt.show()

## Cell 15 — Bedrooms and rent

A boxplot compares rent for different numbers of bedrooms.

This helps us see whether rent generally changes when the number of bedrooms changes.

In [ ]:
plt.figure(figsize=(12,5))

sns.boxplot(x='bedroom', y='price', data=rent)

plt.title('Relation Between Bedrooms and Rent')
plt.xlabel('Bedrooms')
plt.ylabel('Monthly Rent')

plt.show()

## Cell 16 — Area and rent

A scatter plot lets us see the relationship between area and rent.

Each point represents a listing.

In [ ]:
plt.figure(figsize=(12,5))

sns.scatterplot(x='area', y='price', data=rent)

plt.title('Relation Between Area and Monthly Rent')
plt.xlabel('Area (sq.ft)')
plt.ylabel('Monthly Rent')

plt.show()

## Cell 17 — Furnishing type and rent

This boxplot compares rent for:
- Furnished
- Semi-Furnished
- Unfurnished

In [ ]:
plt.figure(figsize=(12,5))

sns.boxplot(x='furnish_type', y='price', data=rent)

plt.title('Furnishing Type vs Monthly Rent')
plt.xlabel('Furnishing Type')
plt.ylabel('Monthly Rent')

plt.show()

## Cell 18 — Property type and rent

This helps us compare rental prices for different property types.

In [ ]:
plt.figure(figsize=(14,5))

sns.boxplot(x='property_type', y='price', data=rent)

plt.title('Property Type vs Monthly Rent')
plt.xlabel('Property Type')
plt.ylabel('Monthly Rent')
plt.xticks(rotation=45)

plt.show()

## Cell 19 — Top 10 localities

There are many localities in the dataset, so we display the 10 localities with the highest number of listings.

`value_counts()` counts each locality and `head(10)` keeps the top 10.

In [ ]:
top_localities = rent['locality'].value_counts().head(10)

print(top_localities)

## Cell 20 — Seller type and rent

We compare the rent distributions for different seller types.

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(x='seller_type', y='price', data=rent)

plt.title('Seller Type vs Monthly Rent')
plt.xlabel('Seller Type')
plt.ylabel('Monthly Rent')

plt.show()

## Cell 21 — Correlation after encoding categories

Linear Regression needs numerical input.

The course notebook uses `LabelEncoder` for categorical columns. We follow the same approach here.

We convert:
- seller_type
- layout_type
- property_type
- locality
- furnish_type

into numbers.

This is done because it is the encoding method taught in the provided housing notebook.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create separate encoders for each categorical column
le_seller = LabelEncoder()
le_layout = LabelEncoder()
le_property = LabelEncoder()
le_locality = LabelEncoder()
le_furnish = LabelEncoder()

# Convert text categories into numbers
rent['seller_type'] = le_seller.fit_transform(rent['seller_type'])
rent['layout_type'] = le_layout.fit_transform(rent['layout_type'])
rent['property_type'] = le_property.fit_transform(rent['property_type'])
rent['locality'] = le_locality.fit_transform(rent['locality'])
rent['furnish_type'] = le_furnish.fit_transform(rent['furnish_type'])

rent.head()

## Cell 22 — Correlation matrix

Now that the input columns are numeric, we can calculate the correlation matrix and display it as a heatmap.

A value closer to `1` means a stronger positive relationship, while a value closer to `-1` means a stronger negative relationship.

Because categorical columns were converted to label numbers, this heatmap should be used as a simple classroom EDA view rather than as a proof of a real numeric relationship.

In [ ]:
cor_mat = rent.corr()

cor_mat

In [ ]:
plt.figure(figsize=(16,6))

plt.title('Relation Between Input and Output Variables')

sns.heatmap(cor_mat, annot=True, cmap='coolwarm')

plt.show()

## Cell 23 — Select X and y

This is the most important step before machine learning.

- **X** = input/features used to make the prediction.
- **y** = target/output we want to predict.

Here the target is `price` because our project predicts monthly rent.

In [ ]:
# Select input columns
X = rent.drop('price', axis=1)

# Select target column
y = rent['price']

print('X columns:')
print(X.columns)

print('\ny shape:', y.shape)
print('X shape:', X.shape)

## Cell 24 — Split the data

We learned `train_test_split()`.

We use:
- 80% of the data for training.
- 20% for testing.

`random_state=42` makes the split repeatable.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

## Cell 25 — Create the Linear Regression model

Linear Regression is the algorithm used in the provided housing notebook.

We create the model first. Training happens in the next cell.

In [ ]:
from sklearn.linear_model import LinearRegression

reg = LinearRegression()

## Cell 26 — Train the model

`fit()` means the model learns the relationship between the input features (`X_train`) and the actual rent (`y_train`).

In [ ]:
reg.fit(X_train, y_train)

print('Model training completed.')

## Cell 27 — See the model coefficients

`reg.coef_` shows the coefficients learned by the Linear Regression model.

There is one coefficient for each input column.

In [ ]:
print('Input columns:')
print(X.columns)

print('\nCoefficients:')
print(reg.coef_)

## Cell 28 — Predict the test data

The trained model now predicts rent for the test data.

`predict()` gives the estimated rent for each test row.

In [ ]:
y_pred = reg.predict(X_test)

print('First 10 actual rents:')
print(y_test.head(10).values)

print('\nFirst 10 predicted rents:')
print(y_pred[:10])

## Cell 29 — Model evaluation

We use the same three regression metrics shown in the provided housing notebook.

### MAE
Mean Absolute Error. It tells us the average absolute prediction error.

### RMSE
Root Mean Squared Error. Large errors have more effect on this value.

### R²
Shows how much of the variation in the target is explained by the model.

For MAE and RMSE, **lower is better**.

For R², **higher is better**.

In [ ]:
from sklearn import metrics

mae = metrics.mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))
r2 = metrics.r2_score(y_test, y_pred)

print('MAE :', round(mae, 2))
print('RMSE:', round(rmse, 2))
print('R2 Score:', round(r2 * 100, 2), '%')

## Cell 30 — Residual plot

Residual = Actual value - Predicted value.

This histogram shows the distribution of prediction errors.

In [ ]:
plt.figure(figsize=(14,6))

sns.histplot(y_test - y_pred, bins=30, color='green')

plt.title('Distribution of Prediction Errors')
plt.xlabel('Actual Rent - Predicted Rent')
plt.ylabel('Frequency')

plt.show()

## Cell 31 — Actual vs predicted values

This scatter plot compares:
- actual rent on the x-axis,
- predicted rent on the y-axis.

If predictions are reasonably close to actual values, the points should generally follow an upward pattern.

In [ ]:
plt.figure(figsize=(14,6))

plt.scatter(y_test, y_pred, color='blue')

plt.title('Actual Rent vs Predicted Rent')
plt.xlabel('Actual Rent')
plt.ylabel('Predicted Rent')

plt.show()

## Cell 32 — Example prediction

We will make one example prediction using the first row from the test set.

`reshape(1,-1)` changes the row into the 2D shape expected by the model.

In [ ]:
sample = X_test.iloc[0].values
sample = np.asarray(sample)
sample = sample.reshape(1, -1)

sample_prediction = reg.predict(sample)

print('Predicted rent for the sample property:', round(sample_prediction[0], 2))

# What we learned from this project

### EDA
We used:
- `head()`
- `tail()`
- `info()`
- `describe()`
- `isnull().sum()`
- histograms
- countplot
- boxplot
- scatterplot
- correlation heatmap

### Data cleaning
We:
- removed duplicate rows,
- cleaned comma-formatted prices,
- converted lakh-format prices,
- cleaned the bathroom column,
- filled missing bathroom values with the median.

### Machine Learning
We used:
- `LabelEncoder`
- `train_test_split`
- `LinearRegression`
- `fit()`
- `predict()`
- MAE
- RMSE
- R²

### Final application
The separate `app.py` uses the same steps and lets a user enter property details to estimate the monthly rent.